In [1]:
import os
import json
import glob
from PIL import Image
import xml.etree.ElementTree as ET

# Configuration
MANIFEST_PATH = 'layout_manifest.json'
TEMPLATE_PATH = 'pathoSlideTemp.svg'
ASSETS_DIR = 'assets'
SLIDE_ID = '001'
OUTPUT_SVG = f'final_slide_output_{SLIDE_ID}.svg'
OVERLAP_THRESHOLD = 0.05 # 5%

def process_assets_and_update_manifest(manifest):
    """Converts images to WebP and updates the manifest references."""
    print("--- Phase 1: Asset Pipeline ---")
    if not os.path.exists(ASSETS_DIR):
        print(f"Directory '{ASSETS_DIR}' not found. Skipping image conversion.")
        return manifest

    image_files = glob.glob(os.path.join(ASSETS_DIR, '*.png')) + glob.glob(os.path.join(ASSETS_DIR, '*.jpg')) + glob.glob(os.path.join(ASSETS_DIR, '*.jpeg'))
    
    for img_path in image_files:
        base_name = os.path.splitext(img_path)[0]
        webp_path = f"{base_name}.webp"
        
        try:
            with Image.open(img_path) as img:
                img.save(webp_path, 'webp', quality=85, optimize=True)
                print(f"Converted: {img_path} -> {webp_path}")
        except Exception as e:
            print(f"Error converting {img_path}: {e}")

    # Update manifest references
    for element in manifest.get('elements', []):
        if element.get('object_type') == 'image' and 'href' in element.get('geometry_attrs', {}):
            old_href = element['geometry_attrs']['href']
            if old_href.endswith(('.png', '.jpg', '.jpeg')):
                new_href = os.path.splitext(old_href)[0] + '.webp'
                element['geometry_attrs']['href'] = new_href
                print(f"Manifest Updated: {old_href} -> {new_href}")
                
    return manifest

def get_bounding_box(element):
    """Calculates a rudimentary Axis-Aligned Bounding Box (AABB) for elements."""
    geo = element.get('geometry_attrs', {})
    obj_type = element.get('object_type')
    
    if obj_type in ['rect', 'image']:
        return {
            'x1': float(geo.get('x', 0)),
            'y1': float(geo.get('y', 0)),
            'x2': float(geo.get('x', 0)) + float(geo.get('width', 0)),
            'y2': float(geo.get('y', 0)) + float(geo.get('height', 0))
        }
    elif obj_type == 'circle':
        cx, cy, r = float(geo.get('cx', 0)), float(geo.get('cy', 0)), float(geo.get('r', 0))
        return {'x1': cx - r, 'y1': cy - r, 'x2': cx + r, 'y2': cy + r}
    elif obj_type == 'text':
        # Approximation for text bounding box based on font-size and rough character width
        x, y = float(geo.get('x', 0)), float(geo.get('y', 0))
        # Default rough bounds if parsing fails
        return {'x1': x - 100, 'y1': y - 50, 'x2': x + 100, 'y2': y + 10}
    
    return None # Paths and complex shapes skipped for basic AABB

def calculate_overlap_percentage(box1, box2):
    """Calculates the intersection over union (IoU) or intersection over smaller box."""
    if not box1 or not box2: return 0.0
    
    x_left = max(box1['x1'], box2['x1'])
    y_top = max(box1['y1'], box2['y1'])
    x_right = min(box1['x2'], box2['x2'])
    y_bottom = min(box1['y2'], box2['y2'])
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0 # No intersection
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    area1 = (box1['x2'] - box1['x1']) * (box1['y2'] - box1['y1'])
    area2 = (box2['x2'] - box2['x1']) * (box2['y2'] - box2['y1'])
    
    if area1 == 0 or area2 == 0: return 0.0
    
    # Calculate overlap against the smaller object
    return intersection_area / min(area1, area2)

def analyze_layout_overlap(manifest):
    """Checks for overlapping elements and generates an AI feedback string."""
    print("\n--- Phase 3: Bounding Box & Overlap Summary ---")
    elements = manifest.get('elements', [])
    boxes = {el['object_id']: get_bounding_box(el) for el in elements}
    
    overlaps = []
    
    for i in range(len(elements)):
        for j in range(i + 1, len(elements)):
            id1, id2 = elements[i]['object_id'], elements[j]['object_id']
            box1, box2 = boxes[id1], boxes[id2]
            
            overlap_pct = calculate_overlap_percentage(box1, box2)
            if overlap_pct > OVERLAP_THRESHOLD:
                overlaps.append(f"Overlap detected: '{id1}' and '{id2}' ({overlap_pct:.1%} overlap).")
    
    if overlaps:
        print("OVERLAPS DETECTED:")
        for o in overlaps: print(f" - {o}")
        
        # AI Feedback String formatting
        ai_feedback = (
            "LAYOUT_ERROR: The following elements exceed the maximum overlap threshold of 5%.\n"
            + "\n".join(overlaps) +
            "\nACTION REQUIRED: Please recalculate the `geometry_attrs` (x, y, width, height) for these "
            "elements to prevent visual collisions and regenerate the `layout_manifest.json`."
        )
        print("\n--- Phase 4: AI Feedback Loop ---")
        print(ai_feedback)
    else:
        print("Layout optimal: No significant overlaps detected.")

def compile_to_svg(manifest, template_path, output_path):
    """Injects JSON manifest into the SVG."""
    print("\n--- Phase 2: SVG Compilation ---")
    
    elements = sorted(manifest.get('elements', []), key=lambda k: k.get('z_order', 0))
    
    # Check if template exists, if not create a blank root
    if os.path.exists(template_path):
        tree = ET.parse(template_path)
        root = tree.getroot()
        # Handle namespaces if necessary
        nsmap = {'svg': 'http://www.w3.org/2000/svg'}
        ET.register_namespace('', nsmap['svg'])
    else:
        print("Template not found. Creating a blank SVG canvas.")
        root = ET.Element('svg', xmlns="http://www.w3.org/2000/svg", viewBox="0 0 7200 10800", width="7200", height="10800")
        tree = ET.ElementTree(root)

    # Inject elements
    for el in elements:
        obj_type = el.get('object_type')
        node = ET.SubElement(root, obj_type)
        
        node.set('id', el.get('object_id', ''))
        
        for k, v in el.get('geometry_attrs', {}).items():
            node.set(k, str(v))
            
        if 'style_attrs' in el:
            node.set('style', el['style_attrs'])
            
        if 'text' in el:
            node.text = el['text']

    tree.write(output_path, encoding='utf-8', xml_declaration=True)
    print(f"Compilation Complete: Saved to {output_path}")

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, 'r') as f:
            manifest_data = json.load(f)
            
        updated_manifest = process_assets_and_update_manifest(manifest_data)
        compile_to_svg(updated_manifest, TEMPLATE_PATH, OUTPUT_SVG)
        analyze_layout_overlap(updated_manifest)
    else:
        print(f"Error: {MANIFEST_PATH} not found.")

--- Phase 1: Asset Pipeline ---

--- Phase 2: SVG Compilation ---
Compilation Complete: Saved to final_slide_output_001.svg

--- Phase 3: Bounding Box & Overlap Summary ---
OVERLAPS DETECTED:
 - Overlap detected: 'bg_canvas' and 'hero_frame' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'main_title' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'subtitle' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'logistics' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'sec_a_title' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'sec_a_p1_l1' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'sec_a_p1_l2' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'sec_a_p2_l1' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'sec_a_p2_l2' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'flow1_box1' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'flow1_t1' (100.0% overlap).
 - Overlap detected: 'bg_canvas' and 'flow1_box2' (100.0% ov